在 PyTorch 中，state_dict（状态字典）是一个极其核心的概念。简单来说，它是模型内部状态的“快照”，本质上是一个普通的 Python 字典对象。
- <font color='red'>它将模型的每一层与其对应的参数张量进行一一映射，是保存和加载模型参数的关键工具。</font>

以下是对 state_dict 的详细介绍：
## 1. state_dict 包含哪些内容？
需要特别注意的是，state_dict <font color='green'>只存储可学习的参数</font>和特定的缓冲区，它不包含模型的类结构、前向传播逻辑或超参数。具体包括：
- <font color='red'>可学习参数（Parameters）</font>：
  - 如卷积层（nn.Conv2d）、线性层（nn.Linear）中的<font color='red'>权重（weight）和偏置（bias）</font>，它们参与梯度计算与优化。
- <font color='red'>持久化缓冲区（Buffers）</font>：
  - <font color='red'>不参与梯度更新但需要在推理时保留的状态</font>。
    - 最典型的例子是批量归一化（BatchNorm）层中的<font color='red'>滑动平均均值（running_mean）和方差（running_var）。</font>
(注：<font color='blue'>Dropout 等没有可训练参数的层不会出现在 state_dict 中。</font>)

## 2. 键名（Keys）的命名规则
state_dict 的键（Key）精确反映了模块的属性层级结构，通常遵循 module.submodule.param_name 的格式。例如：
- 第一层卷积的权重可能表示为：conv1.weight
- 嵌套结构中的 BatchNorm 层可能表示为：layer1.0.bn2.running_mean
这种设计实现了参数与网络结构的解耦，使得状态管理变得高度可预测且易于调试。

## 3. 如何查看 state_dict？
你可以直接调用模型的 state_dict() 方法来查看其内容：

In [ ]:
# 打印模型所有的键名
print(model.state_dict().keys()) 

# 遍历并打印每个参数的名称和形状
for name, param in model.state_dict().items():
    print(f"Layer: {name} | Shape: {param.size()}")

## 4. state_dict 的核心用途
由于它是一个简单的 Python 字典，因此非常灵活，主要用于以下场景：
- 模型序列化（保存与加载）：<font color='red'>PyTorch 官方强烈推荐仅保存 state_dict 而非整个模型对象，因为其耦合度低、可移植性高。</font>

In [ ]:
# 保存
torch.save(model.state_dict(), 'model_weights.pth')

# 加载（必须先实例化一个相同结构的空模型）
model = MyModel()
model.load_state_dict(torch.load('model_weights.pth'))

- <font color='green'>跨设备部署</font>：
  - 通过配合 map_location 参数，可以轻松地将原本在 GPU 上训练的模型权重加载到 CPU 上进行推理。
- <font color='green'>微调预训练模型（Transfer Learning）</font>：
  - 当新模型的架构与原模型有细微差别时，可以使用 strict=False 参数来忽略不匹配的键，或者手动过滤字典来实现部分权重的热启动（Warm Start）。
- <font color='green'>保存和优化器状态</font>：
  - 除了模型本身，优化器（torch.optim）也有自己的 state_dict，用于保存动量、学习率调度等状态，这对于实现断点续训至关重要。